# Retrain Model 2 (conditions) on ReactionT5-base with a fixed vocabulary (Kaggle GPU)

`colab/08_train_conditions_model_reactiont5base.ipynb`'s run (`model2_conditions_reactiont5base`)
finished with a healthy training `eval_loss` (0.14) but **0% valid generation on all 2285
test records** (RESULTS.md, "Спроба: хімічно-передтренований базовий чекпоінт").

**Root cause (confirmed by direct tokenizer testing):** `sagawa/ReactionT5v2-retrosynthesis`'s
268-token vocabulary was built purely from SMILES chemistry notation. It has no tokens for
most JSON punctuation (`{`, `}`, `"`, `:`, `,`) or most English letters -- only the ones that
double as SMILES atom symbols survive. Since this task's targets are JSON strings with
English filler text (`"not specified"`), most training targets were silently collapsing into
an unrecoverable `<unk>`-riddled mess. Teacher-forced training loss looked fine because the
model was correctly learning to reproduce the *corrupted* (but self-consistent) targets --
actual autoregressive generation had no valid completions to produce.

**Fix (`scripts/train_conditions_model.py`, `ensure_full_char_coverage`):** before training,
scan the full train+val corpus, `tokenizer.add_tokens()` any character that maps to `<unk>`,
and `model.resize_token_embeddings()` to match. Verified locally: 65 new single-character
tokens added (vocab 268->303), 0 remaining `<unk>` on the exact JSON target string that
previously produced 24 `<unk>` tokens out of 56.

**Single GPU, not DDP.** A first attempt at this run used `torchrun --nproc_per_node=2`
(the DDP recipe proven for Model 1) and trained without error, but `eval_loss` got stuck
around 8.7-9 after 3+ epochs -- worse than `ln(vocab_size)=5.7`, i.e. worse than guessing
uniformly at random. A local single-process repro of the exact same resize+train code
(CPU, 15 steps) converged normally (loss 16.1->5.1), so the DDP run itself is the
suspect, not the vocab fix or the resize logic. Rather than debug the DDP interaction
further, this notebook just runs single-GPU `python` (drops `torchrun`), matching the
original Colab run's setup exactly (which converged fine, `eval_loss` 0.14) --
same `data/v2_ord_train/conditions_{train,val}.jsonl` (41,139 / 2,285), same `lr=5e-5`.

**Run this as Save & Run All (Commit), not an interactive Draft Session.**

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and
**GPU accelerator**. Kaggle's free GPU quota is **30 hours/week**.

**Data:** `kuzmenkooleh/retro-planner-ord-conditions-41k` (uploaded from
`data/v2_ord_train/conditions_{train,val}.jsonl`).


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))
# Only device 0 is used -- this run is intentionally single-GPU (see markdown above).


In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os, glob

dataset_slug = "retro-planner-ord-conditions-41k"  # @param {type:"string"}

candidates = [
    f"/kaggle/input/{dataset_slug}",
    f"/kaggle/input/datasets/kuzmenkooleh/{dataset_slug}",
]
base = next((c for c in candidates if os.path.exists(os.path.join(c, "conditions_train.jsonl"))), None)
if base is None:
    found = glob.glob("/kaggle/input/**/conditions_train.jsonl", recursive=True)
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "/kaggle/input MISSING"
    assert found, f"conditions_train.jsonl not found under /kaggle/input -- did you add the dataset as an input? /kaggle/input contents: {listing}"
    base = os.path.dirname(found[0])

train_file = os.path.join(base, "conditions_train.jsonl")
val_file = os.path.join(base, "conditions_val.jsonl")
assert os.path.exists(train_file), f"Not found: {train_file}"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Resolved base:", base)
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model2-vocabfix-checkpoint/checkpoint-NNNN  (full Trainer checkpoint)
# or   /kaggle/input/model2-vocabfix-checkpoint/final           (weights only)

In [ ]:
output_dir = "/kaggle/working/model2_conditions_reactiont5base_vocabfix"  # @param {type:"string"}
time_budget_minutes = 100  # @param {type:"number"}
base_model = "sagawa/ReactionT5v2-retrosynthesis"  # @param {type:"string"}
# Single GPU (not DDP -- see markdown above), same recipe as the Colab run that
# converged correctly (eval_loss 0.14): finished in ~1h20min there, so 170 min
# leaves comfortable headroom here too.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!python scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --learning-rate 5e-5 \
    --per-device-train-batch-size 16 \
    --per-device-eval-batch-size 16 \
    --gradient-accumulation-steps 2 \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_work \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the Colab notebook: printing per-step output directly in the cell can make the tab unresponsive over a multi-hour run. Since this runs as a Commit job, you don't need to watch it at all -- check back later via `kaggle kernels status <user>/<slug>` (CLI) or the Output tab.

`--local-work-dir` points at `/kaggle/temp` (fast local scratch disk, wiped between sessions) so Trainer's own checkpoint rotation never touches `/kaggle/working` directly; `scripts/train_conditions_model.py`'s `DriveSyncCallback` still copies out one `latest_checkpoint` folder under `output_dir` after every save.

**When done:** download via CLI (`kaggle kernels output <user>/<slug> -p <dest>`) or the Output tab. `output_dir/final` has the model (with the extended, `<unk>`-free vocabulary already baked in -- no extra step needed to evaluate it). Evaluate it exactly like the previous attempt:

```
python scripts/evaluate_conditions_model_topk.py \
    --test-file data/v2_ord_train/conditions_test.jsonl \
    --model-dir <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model2_topk/conditions_reactiont5base_vocabfix_topk.json
```
